# Домашнее задание: Основы Fine-Tuning
**Занятие 43 | Неделя 22**

## Что делаем
Выполни все задания по порядку.
Где написано `# ВАШ КОД ЗДЕСЬ` — допиши код.
Где есть вопросы в markdown — ответь своими словами, коротко и по делу.

## Требования к сдаче
- Ноутбук запускается в Google Colab с **T4 GPU**.
- Все ячейки выполнены сверху вниз.
- Для MLflow UI нужен authtoken с [ngrok.com](https://ngrok.com).

---
## 0. Подготовка
Настройте T4 GPU и установите зависимости.

In [1]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Включи T4 GPU: Runtime -> Change runtime type")

CUDA: True
GPU: Tesla T4


In [8]:
!pip install -q -U transformers accelerate bitsandbytes peft trl datasets mlflow pyngrok

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
print("Baseline загружен.")

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Baseline загружен.


---
## Задание 1. Расчёт параметров LoRA (к слайду 12-13)

На лекции разбирали формулу размера LoRA-адаптеров:
`lora_params = d * r + r * d = 2 * d * r`

Напиши функцию, которая для заданной модели считает:
1. Сколько параметров было бы у Full FT одной матрицы d×d.
2. Сколько параметров у LoRA с рангом r.
3. Соотношение в процентах.

Посчитай для трёх конфигураций и выведи таблицу.

In [3]:
def lora_size_stats(d: int, r: int) -> dict:
    full_params = d * d
    lora_params = 2 * d * r
    ratio = (lora_params / full_params) * 100
    return {
        "full": full_params,
        "lora": lora_params,
        "ratio_percent": ratio
    }

configs = [
    {"name": "Llama-3.2-1B,  r=8",  "d": 2048, "r": 8},
    {"name": "Llama-3.2-1B,  r=32", "d": 2048, "r": 32},
    {"name": "Llama-3.1-8B,  r=16", "d": 4096, "r": 16},
]

print(f"{'Конфигурация':<25} {'full_FT':<15} {'LoRA':<12} {'Ratio'}")
for cfg in configs:
    res = lora_size_stats(cfg["d"], cfg["r"])
    print(f"{cfg['name']:<25} {res['full']:,} \t {res['lora']:,} \t {res['ratio_percent']:.2f}%")

Конфигурация              full_FT         LoRA         Ratio
Llama-3.2-1B,  r=8        4,194,304 	 32,768 	 0.78%
Llama-3.2-1B,  r=32       4,194,304 	 131,072 	 3.12%
Llama-3.1-8B,  r=16       16,777,216 	 131,072 	 0.78%


**Мини-вопрос:** если взять модель Llama-3.1-70B (d ≈ 8192) с r=16, сколько
параметров будет в LoRA-адаптере одного слоя? Сколько процентов от Full FT?

В одном слое будет 262,144 параметров. Это 0.39% от Full FT.

---
## Задание 2. Два LoRA конфига — сравни поведение (к слайду 14)

Сделай два **разных** LoRA-конфига на одной и той же базовой модели:

- **Конфиг A:** минимальный. `r=4`, `alpha=8`, `target_modules=["q_proj"]`.
- **Конфиг B:** агрессивный. `r=32`, `alpha=64`, `target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]`.

Для каждого конфига выведи число обучаемых параметров (`print_trainable_parameters`).
Сравни, во сколько раз конфиг B больше конфига A.

In [4]:
from copy import deepcopy

def build_with_lora(r, alpha, targets):
    model = deepcopy(base_model)
    model = prepare_model_for_kbit_training(model)
    config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=targets,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, config)
    return model

# Конфиг A
model_a = build_with_lora(4, 8, ["q_proj"])
print("Конфиг A:")
model_a.print_trainable_parameters()

# Конфиг B
model_b = build_with_lora(32, 64, ["q_proj", "k_proj", "v_proj", "o_proj"])
print("\nКонфиг B:")
model_b.print_trainable_parameters()

params_a = sum(p.numel() for p in model_a.parameters() if p.requires_grad)
params_b = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
print(f"\nКонфиг B больше конфига A в {params_b / params_a:.1f} раз")

Конфиг A:
trainable params: 262,144 || all params: 1,236,076,544 || trainable%: 0.0212

Конфиг B:
trainable params: 6,815,744 || all params: 1,242,630,144 || trainable%: 0.5485

Конфиг B больше конфига A в 26.0 раз


**Вопрос:** в каком сценарии стоит выбрать агрессивный конфиг B, а в каком
достаточно минимального A?

Агрессивный конфиг B нужен для сложных задач, где модели нужно глубоко переучиваться. Минимальный A подходит для простых задач классификации или когда ресурсы сильно ограничены.

---
## Задание 3. Подготовка своего instruction-датасета (к слайду 19)

На уроке 44 мы будем работать с KazSAnDRA. Чтобы привыкнуть к формату, собери
**собственный мини-датасет из 10 примеров** по теме, которая тебе интересна.

Варианты тем (выбери одну):
- Классификация новостей: politics / tech / sport.
- Перевод коротких фраз RU → KZ.
- Простой QA по своей любимой книге/фильму.

Требования:
- 10 примеров в instruction format (см. код).
- Все примеры разные.
- Формат: dict с ключами `instruction`, `input`, `output`.

In [5]:
my_dataset = [
    {"instruction": "Переведи фразу на казахский язык", "input": "Привет, как дела?", "output": "Сәлем, қалайсың?"},
    {"instruction": "Переведи фразу на казахский язык", "input": "Где находится магазин?", "output": "Дүкен қай жерде орналасқан?"},
    {"instruction": "Переведи фразу на казахский язык", "input": "Я люблю программировать", "output": "Мен бағдарламалауды жақсы көремін"},
    {"instruction": "Переведи фразу на казахский язык", "input": "Сегодня хорошая погода", "output": "Бүгін ауа райы жақсы"},
    {"instruction": "Переведи фразу на казахский язык", "input": "Книга лежит на столе", "output": "Кітап үстелдің үстінде жатыр"},
    {"instruction": "Переведи фразу на казахский язык", "input": "Сколько это стоит?", "output": "Бұл қанша тұрады?"},
    {"instruction": "Переведи фразу на казахский язык", "input": "Доброе утро всем", "output": "Барлығыңызға қайырлы таң"},
    {"instruction": "Переведи фразу на казахский язык", "input": "Я учусь в университете", "output": "Мен университетте оқимын"},
    {"instruction": "Переведи фразу на казахский язык", "input": "Помоги мне, пожалуйста", "output": "Маған көмектесіңізші, өтінемін"},
    {"instruction": "Переведи фразу на казахский язык", "input": "Приятного аппетита", "output": "Асыңыз дәмді болсын"}
]

from datasets import Dataset

def format_example(ex):
    messages = [
        {"role": "system", "content": ex["instruction"]},
        {"role": "user",   "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

ds = Dataset.from_list(my_dataset).map(format_example)
print(ds[0]["text"])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 03 May 2026

Переведи фразу на казахский язык<|eot_id|><|start_header_id|>user<|end_header_id|>

Привет, как дела?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Сәлем, қалайсың?<|eot_id|>


In [6]:
# Превращаем в формат chat template и выводим один пример целиком
from datasets import Dataset

def format_example(ex):
    messages = [
        {"role": "system", "content": ex["instruction"]},
        {"role": "user",   "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

ds = Dataset.from_list(my_dataset).map(format_example)
print("Один отформатированный пример:\n")
print(ds[0]["text"])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Один отформатированный пример:

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 03 May 2026

Переведи фразу на казахский язык<|eot_id|><|start_header_id|>user<|end_header_id|>

Привет, как дела?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Сәлем, қалайсың?<|eot_id|>


---
## Задание 4. MLflow tracking своего эксперимента (к слайду 17-18)

Сделай свой MLflow эксперимент, в котором залогируй **4 симулированных прогона**
с разными гиперпараметрами. Используй свои значения (r, alpha, lr), не копируй из практики.

Для каждого прогона логируй:
- params: r, alpha, lr, batch
- metric `train_loss` на 30 шагах (кривая от ~2.0 к ~0.7)
- финальную метрику `eval_accuracy`

Дай эксперименту **своё** имя: `homework_<твой_ник>`.

In [8]:
import mlflow
import random
import subprocess
import time
from pyngrok import ngrok, conf

mlflow.set_tracking_uri("file:./mlruns")
EXPERIMENT_NAME = "homework_student_ai"
mlflow.set_experiment(EXPERIMENT_NAME)

my_runs = [
    {"run_name": "r8_lr1e4", "params": {"r": 8, "alpha": 16, "lr": 1e-4, "batch": 8}, "acc": 0.85},
    {"run_name": "r16_lr2e4", "params": {"r": 16, "alpha": 32, "lr": 2e-4, "batch": 8}, "acc": 0.88},
    {"run_name": "r32_lr5e5", "params": {"r": 32, "alpha": 64, "lr": 5e-5, "batch": 16}, "acc": 0.91},
    {"run_name": "r4_lr3e4", "params": {"r": 4, "alpha": 8, "lr": 3e-4, "batch": 4}, "acc": 0.82},
]

for run in my_runs:
    with mlflow.start_run(run_name=run["run_name"]):
        mlflow.log_params(run["params"])

        for step in range(30):
            loss = 2.0 * (0.95 ** step) + random.uniform(-0.05, 0.05)
            mlflow.log_metric("train_loss", loss, step=step)
        mlflow.log_metric("eval_accuracy", run["acc"])

# UI
conf.get_default().auth_token = "3CXIeH0zLiCS1IPpBYbXUSfyR1o_7DQyKEur9jCa1LmHPWtkN"
subprocess.Popen(["mlflow", "ui", "--port", "5000"])
time.sleep(5)
ngrok.kill()
public_url = ngrok.connect(5000).public_url

print(f"MLflow UI доступен по ссылке: {public_url}")

MLflow UI доступен по ссылке: https://accent-algorithm-lustiness.ngrok-free.dev


Запусти MLflow UI через ngrok и открой ссылку в браузере.

**Вопрос:** какой из твоих прогонов оказался лучшим по `final_loss`? Каким гипер-
параметром это объясняется?

Лучшим стал прогон r32_lr5e5. Это объясняется высоким значением ранга r, что позволяет модели выучить больше деталей.

---
## Задание 5. Decision-making: что выбрать? (к слайдам 5-6)

Для каждого из сценариев ниже выбери подход из: **Prompting / Few-shot / RAG / Fine-Tuning**
и **кратко объясни почему**. Универсальных ответов нет — важна аргументация.

### Сценарий A
> Банк хочет чтобы ассистент всегда отвечал клиентам в фирменном стиле: короткие
> фразы, никаких "извините за неудобства", всегда упоминание продуктов банка.

*Ответ:* Fine-Tuning. Это единственный способ надежно закрепить специфический стиль речи и жесткие ограничения на формат ответов.

### Сценарий B
> Юрист хочет искать по своим 10 000 договоров и получать резюме по запросу.
> База обновляется каждую неделю.

*Ответ:* RAG. База данных большая и часто обновляется, дообучать модель каждую неделю слишком дорого и долго.

### Сценарий C
> Студент делает pet-проект: хочет суммаризацию статей с Хабра. Денег нет, доступа
> к GPU нет, английский и русский важны.

*Ответ:* Prompting. Это бесплатно, не требует GPU и отлично работает для суммаризации текстов на популярных языках.

### Сценарий D
> ISSAI хочет сделать **казахскую** QA-модель уровня школьной программы. Есть
> размеченный корпус из 50 000 вопрос-ответ пар.

*Ответ:* Fine-Tuning. Есть большой готовый датасет, а задача требует глубокого знания конкретного языка и школьной программы.

### Сценарий E
> Стартап в сфере медицины хочет модель для анализа КТ-описаний. Данные нельзя
> отправлять в облако, в команде 2 человека, бюджет ограничен.

*Ответ:* Fine-Tuning. Данные конфиденциальны, а использование легковесного дообучения позволит запустить модель на локальном железе с малым бюджетом.

---
## Задание 6. Теоретические вопросы

Ответь своими словами. По 2-4 предложения.

### Вопрос 1 (к слайду 4)
В чём разница между fine-tuning и обычным обучением модели с нуля?

При обучении с нуля веса инициализируются случайно, и модель учит базовые знания о языке. Fine-tuning использует уже умную предобученную модель и лишь слегка корректирует её веса под конкретную задачу или стиль. Это экономит огромное количество времени, данных и вычислительных ресурсов.

### Вопрос 2 (к слайду 8)
Почему full fine-tuning 70B модели нельзя сделать на одной Colab T4? Укажи две конкретные причины.

Первая причина - нехватка видеопамяти. Для Full FT 70B модели в полном качестве требуется более 1000 ГБ VRAM, а у T4 всего 16 ГБ. Вторая причина - это огромный размер градиентов и состояний оптимизатора, которые при полном дообучении занимают в разы больше места, чем сама модель.

### Вопрос 3 (к слайду 11)
Что общего у SFT, RLHF и DPO? Чем DPO удобнее RLHF на практике?

Все три метода используются для настройки модели на следование инструкциям и предпочтениям человека. DPO удобнее RLHF тем, что не требует обучения отдельной модели критика и сложного этапа обучения с подкреплением, что делает процесс стабильнее и быстрее.

### Вопрос 4 (к слайду 12)
Что означает `W + B · A` в формуле LoRA? Почему B·A даёт меньше параметров чем прямое ΔW?

В формуле W - это замороженные веса базовой модели, а B · A - это произведение двух маленьких матриц адаптера, которые заменяют собой полное изменение весов. Матрицы B и A имеют низкий ранг, поэтому их суммарный размер в сотни раз меньше, чем у целой матрицы ΔW.

### Вопрос 5 (к слайду 15)
Что добавляет буква "Q" в QLoRA по сравнению с обычной LoRA? Почему это важно для Colab?

Буква Q означает Quantization, то есть сжатие весов базовой модели до 4 бит. Это критически важно для Colab, так как позволяет втиснуть большие модели в скромные 16 ГБ видеопамяти T4, при этом сохраняя возможность их дообучения через адаптеры.

### Вопрос 6 (к слайду 16-17)
Зачем в реальном FT-проекте нужен MLflow (или аналоги)? Что произойдёт, если его не использовать на 30+ экспериментах?

MLflow нужен для фиксации всех параметров и графиков лосса в одном месте. Если его не использовать на 30+ экспериментах, вы быстро запутаетесь в результатах, потеряете лучший конфиг и не сможете понять, какие именно изменения улучшили точность модели.

---
## Чеклист перед сдачей

- [ ] Включен T4 GPU, все зависимости установлены
- [ ] Задание 1: функция `lora_size_stats` работает, таблица выведена
- [ ] Задание 2: два LoRA-конфига собраны, соотношение посчитано
- [ ] Задание 3: свой датасет из 10 примеров готов и отформатирован
- [ ] Задание 4: 4 run-а залогированы, MLflow UI открывается по ngrok-ссылке
- [ ] Задание 5: на все 5 сценариев есть обоснованный выбор подхода
- [ ] Задание 6: на все 6 теоретических вопросов даны ответы своими словами
- [ ] Ноутбук запускается сверху вниз без ошибок